In [1]:
pwd

'/rds/general/user/bc1721/home/actress/actress'

In [2]:
'''
import actress.contrasts as contrst

for i in range(11):   # 0 → 9 (use range(11) if you want 0–10)

    fac_file  = f'./inputs/limb_files/mean300_{i}.txt'
    phot_file = f'./inputs/limb_files/meanSSD_{i}.txt'

    save_fac  = f'test_fac_{i}.txt'
    save_phot = f'test_phot_{i}.txt'

    print(f"Processing index {i}")

    contrst.contrast(
        fac_file,
        wav_lower=1.8050e-7,
        wav_higher=2.9950e-7,
        N=120,
        ld_law='power2',
        save=save_fac,
        graphs=False
    )

    contrst.contrast(
        phot_file,
        wav_lower=1.8050e-7,
        wav_higher=2.9950e-7,
        N=120,
        ld_law='power2',
        save=save_phot,
        graphs=False
    )
'''

'\nimport actress.contrasts as contrst\n\nfor i in range(11):   # 0 → 9 (use range(11) if you want 0–10)\n\n    fac_file  = f\'./inputs/limb_files/mean300_{i}.txt\'\n    phot_file = f\'./inputs/limb_files/meanSSD_{i}.txt\'\n\n    save_fac  = f\'test_fac_{i}.txt\'\n    save_phot = f\'test_phot_{i}.txt\'\n\n    print(f"Processing index {i}")\n\n    contrst.contrast(\n        fac_file,\n        wav_lower=1.8050e-7,\n        wav_higher=2.9950e-7,\n        N=120,\n        ld_law=\'power2\',\n        save=save_fac,\n        graphs=False\n    )\n\n    contrst.contrast(\n        phot_file,\n        wav_lower=1.8050e-7,\n        wav_higher=2.9950e-7,\n        N=120,\n        ld_law=\'power2\',\n        save=save_phot,\n        graphs=False\n    )\n'

In [3]:
'''
import actress.contrasts as contrst


fac_file = './inputs/limb_files/mean300.txt'
#fac_file = './inputs/limb_files/limb_atlkurab_vt1_K0_100Gmol_means.txt'
phot_file = './inputs/limb_files/meanSSD.txt'

#Jean's line
contrst.contrast(fac_file, wav_lower=1.8050e-7, wav_higher=2.9950e-7, N=120, ld_law='power2', save='test_fac_0.txt', graphs=False)
contrst.contrast(phot_file, wav_lower=1.8050e-7, wav_higher=2.9950e-7, N=120, ld_law='power2', save='test_phot_0.txt', graphs=False)

#Jean's 2nd line
# contrst.contrast(fac_file, wav_lower=4.37935e-7, wav_higher=4.37965e-7, N=20, ld_law='power2', save='test_fac2.txt', graphs=False)
# contrst.contrast(phot_file,  wav_lower=4.37935e-7, wav_higher=4.37965e-7, N=20, ld_law='power2', save='test_phot2.txt', graphs=False)

#Continuum
#contrst.contrast(fac_file, wav_lower=4.37835e-7, wav_higher=4.37865e-7, N=20, ld_law='power2', save='test_fac2.txt', graphs=False)
#contrst.contrast(phot_file,  wav_lower=4.37835e-7, wav_higher=4.37865e-7, N=20, ld_law='power2', save='test_phot2.txt', graphs=False)

#contrst.teff(fac_file, ld_law=None) #instead of this function, do the inverse planck at this function and divide by wavelength bin width to get representative flux at that wavelength, to get the temperature
#contrst.teff(phot_file, ld_law='None')
#contrst.teff(fac_file, ld_law='custom')
#contrst.teff(phot_file, ld_law='custom')
'''

"\nimport actress.contrasts as contrst\n\n\nfac_file = './inputs/limb_files/mean300.txt'\n#fac_file = './inputs/limb_files/limb_atlkurab_vt1_K0_100Gmol_means.txt'\nphot_file = './inputs/limb_files/meanSSD.txt'\n\n#Jean's line\ncontrst.contrast(fac_file, wav_lower=1.8050e-7, wav_higher=2.9950e-7, N=120, ld_law='power2', save='test_fac_0.txt', graphs=False)\ncontrst.contrast(phot_file, wav_lower=1.8050e-7, wav_higher=2.9950e-7, N=120, ld_law='power2', save='test_phot_0.txt', graphs=False)\n\n#Jean's 2nd line\n# contrst.contrast(fac_file, wav_lower=4.37935e-7, wav_higher=4.37965e-7, N=20, ld_law='power2', save='test_fac2.txt', graphs=False)\n# contrst.contrast(phot_file,  wav_lower=4.37935e-7, wav_higher=4.37965e-7, N=20, ld_law='power2', save='test_phot2.txt', graphs=False)\n\n#Continuum\n#contrst.contrast(fac_file, wav_lower=4.37835e-7, wav_higher=4.37865e-7, N=20, ld_law='power2', save='test_fac2.txt', graphs=False)\n#contrst.contrast(phot_file,  wav_lower=4.37835e-7, wav_higher=4.3786

In [4]:
import actress as ac

def compute_ff(rs, longs, lats):
    sim = ac.Simulator()
    for i in range(len(rs)):
        sim.addfeature(r=rs[i], lon=longs[i], lat=lats[i], feature='fac')
    return sim.getfill('fac', mode='faconly')#, sim_ff.getdiscfill('fac', mode='faconly', inc=90)


def rescale_rs_to_target_ff(rs, longs, lats, target_ff, tol=1e-4, max_iter=20):
    
    ff0 = compute_ff(rs, longs, lats)
    
    if ff0 == 0:
        raise ValueError("Initial FF is zero — cannot rescale")
    
    s = np.sqrt(target_ff / ff0)
    
    for _ in range(max_iter):
        rs_scaled = rs * s
        ff = compute_ff(rs_scaled, longs, lats)
        
        if abs(ff - target_ff) < tol:
            print(f"Converged: FF = {ff:.5f}")
            return rs_scaled
        
        s *= np.sqrt(target_ff / ff)
    
    print(f"Warning: final FF = {ff:.5f}")
    return rs_scaled

In [5]:
'''
import numpy as np
import actress
import actress.run_actress_notebook as actr

# -------------------------
# 1. Fixed facula distribution
# -------------------------
np.random.seed(123)  # optional, for reproducibility

Nfac = 250

rs = np.random.lognormal(mean=0.2, sigma=0.3, size=Nfac)
longs = np.random.normal(180, 90, Nfac) 
lats_raw = np.random.normal(0, 8, Nfac)
lats = np.clip(lats_raw, -90, 90)

# optional: rescale rs here if using target FF
target_ff = 0.05  # e.g. 5% coverage

rs = rescale_rs_to_target_ff(rs, longs, lats, target_ff)

print("Final FF check (Surface FF):", compute_ff(rs, longs, lats))

np.savez("faculae_distribution.npz", rs=rs, longs=longs, lats=lats)
'''

'\nimport numpy as np\nimport actress\nimport actress.run_actress_notebook as actr\n\n# -------------------------\n# 1. Fixed facula distribution\n# -------------------------\nnp.random.seed(123)  # optional, for reproducibility\n\nNfac = 250\n\nrs = np.random.lognormal(mean=0.2, sigma=0.3, size=Nfac)\nlongs = np.random.normal(180, 90, Nfac) \nlats_raw = np.random.normal(0, 8, Nfac)\nlats = np.clip(lats_raw, -90, 90)\n\n# optional: rescale rs here if using target FF\ntarget_ff = 0.05  # e.g. 5% coverage\n\nrs = rescale_rs_to_target_ff(rs, longs, lats, target_ff)\n\nprint("Final FF check (Surface FF):", compute_ff(rs, longs, lats))\n\nnp.savez("faculae_distribution.npz", rs=rs, longs=longs, lats=lats)\n'

In [6]:
import numpy as np
import actress
import actress.run_actress_notebook as actr
# specific combinations
pairs = []

# single case
pairs.append((0, 5))  # phot0 + fac2
#pairs.append((0, 1))
# combinations
'''
phot_indices = [4, 5, 9]
fac_indices  = [3, 5, 9]

for i in phot_indices:
    for j in fac_indices:
        pairs.append((i, j))
'''
data = np.load("faculae_distribution.npz")

rs = data["rs"]
longs = data["longs"]
lats = data["lats"]

data = np.load("faculae_distribution.npz")

rs = data["rs"]
longs = data["longs"]
lats = data["lats"]
# -------------------------
# 2. List your LD files
# -------------------------
phot_files = [
    "test_phot_0.txt",
    "test_phot_1.txt",
    "test_phot_2.txt",
    "test_phot_3.txt",
    "test_phot_4.txt",
    "test_phot_5.txt",
    "test_phot_6.txt",
    "test_phot_7.txt",
    "test_phot_8.txt",
    "test_phot_9.txt",
    "test_phot_10.txt",
]

fac_files = [
    "test_fac_0.txt",
    "test_fac_1.txt",
    "test_fac_2.txt",
    "test_fac_3.txt",
    "test_fac_4.txt",
    "test_fac_5.txt",
    "test_fac_6.txt",
    "test_fac_7.txt",
    "test_fac_8.txt",
    "test_fac_9.txt",
    "test_fac_10.txt",
]

In [ ]:
for i, j in pairs:

    phot_file = phot_files[i]
    fac_file  = fac_files[j]

    params = actr.Transitparams()

    params.res = 200
    params.rp = 0.3808
    params.b = -0.04
    params.N = 100
    params.mode = "faconly"

    params.fac_r = rs
    params.fac_long = longs
    params.fac_lat = lats

    params.fac_band = False
    params.fac_band_low = [40]
    params.fac_band_high = [40]

    params.a = 25.01
    params.T = 4.544
    params.phi = 1.0
    params.ld = "power2"

    save = f"batch_phot{i+1}_fac{j+1}"

    print(f"Running {save}")
    print(f"Phot file: {phot_file}")
    print(f"Fac file:  {fac_file}")

    actr.Transitsim(params).sim_spectrum(
        hd_ld_file=phot_file,
        fac_ld_file=fac_file,
        gif_save=save,
        lightcurve_save=save
    )

Running batch_phot1_fac6
Phot file: test_phot_0.txt
Fac file:  test_fac_5.txt
1805.000 Angstroms
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)


MovieWriter ffmpeg unavailable; using Pillow instead.


v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
1815.000 Angstroms
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)


MovieWriter ffmpeg unavailable; using Pillow instead.


v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
1825.000 Angstroms
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)


...

v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
2985.000 Angstroms
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)


MovieWriter ffmpeg unavailable; using Pillow instead.


v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
2995.000 Angstroms
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)


MovieWriter ffmpeg unavailable; using Pillow instead.


v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)
v2p: functools.partial(<function vec2pix at 0x14556a214c20>, 1024)


In [ ]:
'''
# -------------------------
# 3. Run all phot/fac combinations
# -------------------------
for i, phot_file in enumerate(phot_files):
    for j, fac_file in enumerate(fac_files):

        params = actr.Transitparams()

        params.res = 200
        params.rp = 0.3808
        params.b = -0.04
        params.N = 100
        params.mode = "faconly"

        # same facula distribution every run
        params.fac_r = rs
        params.fac_long = longs
        params.fac_lat = lats

        params.fac_band = False
        params.fac_band_low = [40]
        params.fac_band_high = [40]

        params.a = 25.01
        params.T = 4.544
        params.phi = 1.0
        params.ld = "power2"

        save = f"batch_phot{i+1}_fac{j+1}"

        print(f"Running {save}")
        print(f"Phot file: {phot_file}")
        print(f"Fac file:  {fac_file}")
        if i==9 and j==9:
            actr.Transitsim(params).sim_spectrum(
                hd_ld_file=phot_file,
                fac_ld_file=fac_file,
                gif_save=save,
                lightcurve_save=save
            )
        if i !=9 and j!=9:
            actr.Transitsim(params).sim_spectrum(
                hd_ld_file=phot_file,
                fac_ld_file=fac_file,
                gif_save=False,
                lightcurve_save=save
            )
'''

In [ ]:
'''
import actress #REFRESH BEFORE USE!
import numpy as np
import actress.run_actress_notebook as actr

hd_ld_file = 'test_phot.txt'  #Input Quiet star limb darkening file
fac_ld_file = 'test_fac.txt'  #Input Facular limb darkening file
#fac_ld_file = 'itert3126g4826_ld.txt'

params = actr.Transitparams()

# number of faculae
N = 250
# # facula sizes (log-normal, clipped between 5–15)
rs = np.random.lognormal(mean=0.2, sigma=0.3, size=N)    #angular radius in degree
#rs2 = np.random.lognormal(mean=-1, sigma=0.7, size=60)  
#rs = np.concatenate([rs, rs2])

# # For longitudes: can range from 0 to 2π (0 to 360 degrees)
longs = np.random.normal(180, 90, N)                         # Gaussian longitudes (mean=180, std=45)
# # Wrap longitudes to [0, 360] range
longs = longs % 360

#latitude positve: southern hamesphere
lats_raw = np.random.normal(0, 8, N) 
lats = np.clip(lats_raw, -90, 90)      
print(rs, longs, lats)

target_ff = 0.05  # e.g. 5% coverage

rs = rescale_rs_to_target_ff(rs, longs, lats, target_ff)

print("Final FF check (Surface FF):", compute_ff(rs, longs, lats))



params.res = 200          #Resolution of the simulation box #vs inside simulator.py resolution of healpix map -- explain difference, def takes longer with more resolution but can't tell from gifs??
params.rp = 0.3808            # Rp/R* 
params.b = -0.04               # Transit impact parameter
params.N = 100             # Number of data points in phase resolution
params.mode = 'faconly'      # 'quiet' or 'faconly'

params.fac_r    = rs
params.fac_long = longs
params.fac_lat  = lats
params.fac_band = False     # Add facular bands (True/False)
params.fac_band_low = [40]   # Lower latitude of band
params.fac_band_high = [40]  # Upper latitude of band

params.a = 25.01              # a/R* ?maybe is in actual units?
params.T = 4.544               # Orbital period 
params.phi = 0.9          # Phase coverage
params.ld = 'power2'       # Limb darkening ('power2' or 'quadratic' or 'claret')



save = 'slow_rotate_spot_135long_0lat'  #Name of lightcurve output file
actr.Transitsim(params).sim_spectrum(hd_ld_file=hd_ld_file, fac_ld_file=fac_ld_file, gif_save=save, lightcurve_save=save) #Give descriptive name for run in gif_save and lightcurve_save

'''